# Machine Learning

In [3]:
from collections import Counter
import json

def cargar_datos_entrenamiento(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    X = []
    y = []
    ids = []

    for key, value in data.items():
        tweet_text = value['tweet']
        votos = value['labels_task1']
        
        conteo = Counter(votos)
        label_mayoritaria, frecuencia = conteo.most_common(1)[0]
        total_votos = len(votos)
        if frecuencia <= total_votos / 2:
            continue 
            
        X.append(tweet_text)
        y.append(label_mayoritaria)
        ids.append(value['id_EXIST'])

    return X, y, ids



ruta = 'dataset_task1_exist2025/training.json' 
try:
    X_train, y_train, ids_train = cargar_datos_entrenamiento(ruta)
    
    print(f"Procesados {len(X_train)} tweets correctamente.")
    print(f"Ejemplo - Texto: {X_train[0][:50]}...")
    print(f"Ejemplo - Label: {y_train[0]}")

except FileNotFoundError:
    print("Error: No se encontró el archivo training.json. Revisa la ruta.")

Procesados 6064 tweets correctamente.
Ejemplo - Texto: @TheChiflis Ignora al otro, es un capullo.El probl...
Ejemplo - Label: YES


In [4]:
import numpy as np
import re
import nltk
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


class LimpiadorTweets(BaseEstimator, TransformerMixin):
    def __init__(self, quitar_menciones=True, quitar_urls=True, quitar_hashtags=True, quitar_puntuacion=False):
        self.quitar_menciones = quitar_menciones
        self.quitar_urls = quitar_urls
        self.quitar_hashtags = quitar_hashtags
        self.quitar_puntuacion = quitar_puntuacion

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_limpio = []
        for texto in X:
            texto = texto.lower()
            if self.quitar_urls:
                texto = re.sub(r'https?://\S+|www\.\S+', '', texto)
            if self.quitar_menciones:
                texto = re.sub(r'@[\w]+', '', texto)
            if self.quitar_hashtags:
                texto = texto.replace('#', '')
            if self.quitar_puntuacion:
                texto = re.sub(r'[^\w\s]', '', texto)
            X_limpio.append(' '.join(texto.split()))
        return X_limpio

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')


class POSTagExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        features = []
        for text in X:
            tokens = nltk.word_tokenize(text)
            try:
                tags = nltk.pos_tag(tokens)
            except LookupError:
                # Si falla por el idioma, forzamos el 'eng'
                tags = nltk.pos_tag(tokens, lang='eng')
            
            tag_counts = Counter([tag for word, tag in tags])
            total = len(tokens) if len(tokens) > 0 else 1
            
            features.append([
                tag_counts.get('JJ', 0)/total, 
                tag_counts.get('VB', 0)/total, 
                tag_counts.get('NN', 0)/total
            ])
        return np.array(features)

# Pipeline Principal con FeatureUnion
# Se combina el texto (TF-IDF/BoW) con los POS Tags
pipeline_final = Pipeline([
    ('limpieza', LimpiadorTweets()), 
    ('features', FeatureUnion([
        ('text', TfidfVectorizer()), 
        ('pos', POSTagExtractor())
    ])),
    ('reduccion', 'passthrough'),
    ('clf', LinearSVC(random_state=42, max_iter=5000))
])


param_grid_final = [
    # SVM y Regresión Logística
    {
        'clf': [LinearSVC(random_state=42, max_iter=5000), 
                LogisticRegression(random_state=42, solver='liblinear')],
        'clf__C': [0.1, 1, 10],
        'features__text': [TfidfVectorizer(sublinear_tf=True), CountVectorizer()],
        'features__text__ngram_range': [(1, 2)],
        'reduccion': ['passthrough', TruncatedSVD(n_components=100)],
        'limpieza__quitar_menciones': [True, False],
        'limpieza__quitar_puntuacion': [True, False],
    },
    
    # Random Forest
    {
        'clf': [RandomForestClassifier(random_state=42)],
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [None, 20],
        'features__text': [TfidfVectorizer(sublinear_tf=True)],
        'features__text__ngram_range': [(1, 1)],
        'reduccion': ['passthrough'],
        'limpieza__quitar_menciones': [True],
    }
]

grid_search = GridSearchCV(
    pipeline_final, 
    param_grid_final, 
    cv=5, 
    scoring='f1_macro', 
    verbose=3, 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Mejor F1-score: {grid_search.best_score_:.4f}")
print(f"Mejor configuración: {grid_search.best_params_}")

Fitting 5 folds for each of 100 candidates, totalling 500 fits


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\franm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\franm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\franm\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Mejor F1-score: 0.6537
Mejor configuración: {'clf': LinearSVC(max_iter=5000, random_state=42), 'clf__C': 1, 'features__text': TfidfVectorizer(sublinear_tf=True), 'features__text__ngram_range': (1, 2), 'limpieza__quitar_menciones': False, 'limpieza__quitar_puntuacion': False, 'reduccion': 'passthrough'}


In [5]:
import json
import pandas as pd

print("Reentrenando el mejor modelo con el dataset completo...")
mejor_pipeline = grid_search.best_estimator_
mejor_pipeline.fit(X_train, y_train)


ruta_test = 'dataset_task1_exist2025/test.json'
try:
    with open(ruta_test, 'r', encoding='utf-8') as f:
        test_data = json.load(f)

    ids_test = []
    textos_test = []

    for key, value in test_data.items():
        ids_test.append(value['id_EXIST'])
        textos_test.append(value['tweet'])

    print("Generando predicciones para el test oficial...")
    predicciones_finales = mejor_pipeline.predict(textos_test)

    preds_dict = []
    for id_ev, val in zip(ids_test, predicciones_finales):
        preds_dict.append({
            'test_case': 'EXIST2025',
            'id': id_ev,
            'value': val
        })

    # Guardar archivo
    nombre_archivo = 'Thinkpol_modelo_SVM.json'
    
    with open(nombre_archivo, 'w', encoding='utf-8') as f:
        json.dump(preds_dict, f, ensure_ascii=False, indent=4)

    print(f"Archivo {nombre_archivo} generado con éxito.")
    print(f"Configuración utilizada: {grid_search.best_params_}")

except FileNotFoundError:
    print("Error: No se encontró el archivo test.json.")

Reentrenando el mejor modelo con el dataset completo...
Generando predicciones para el test oficial...
Archivo Thinkpol_modelo_SVM.json generado con éxito.
Configuración utilizada: {'clf': LinearSVC(max_iter=5000, random_state=42), 'clf__C': 1, 'features__text': TfidfVectorizer(sublinear_tf=True), 'features__text__ngram_range': (1, 2), 'limpieza__quitar_menciones': False, 'limpieza__quitar_puntuacion': False, 'reduccion': 'passthrough'}
